# EasyMagpieTTS — vLLM-Omni inference demo (dummy weights)

This notebook runs an end-to-end inference pass through the
[`easymagpie_vllm_omni`](./easymagpie_vllm_omni) model definition using
**dummy / random weights**, so you can exercise the full engine path
(prefill -> autoregressive decode -> audio-code extraction) without a converted
checkpoint.

It follows the same `AsyncOmni` single-stage pattern as the reference
`qwen3-tts` and `eartts` demos:

* **prefill** — the caller supplies a precomputed context embedding via
  `additional_information.prompt_embeds` of shape `(T_ctx, embedding_dim)`, with
  `prompt_token_ids = [0] * T_ctx` (exactly like qwen3-tts `talker_prompt_embeds`
  / eartts `speaker_latent`).
* **decode** — each step consumes one subword id from the streaming
  `additional_information.text_tokens` list; the local transformer samples all
  `C * S` stacked audio codebooks for the frame.
* **output** — per-step audio codes are surfaced on
  `OmniOutput.multimodal_outputs[\"audio_codes\"]` (`BT x num_codebooks`), and the
  engine accumulates them across steps just like eartts, so we trim to the last
  `len(token_ids)` decoded rows.

> **Dummy weights.** We build a tiny `config.json` (small backbone + small
> codebooks) and start the engine with `load_format=\"dummy\"`, so vLLM fills all
> parameters with random values. The emitted codes are therefore meaningless —
> this is a *smoke test* of the engine wiring, not a real synthesis. Point the
> engine at a real converted checkpoint (and drop `load_format`) to get audio.

> **Environment.** Run this inside the bootstrapped `vllm_omni_env` (vLLM +
> vLLM-Omni + compatible torch) with the plugin installed:
> ```bash
> source /path/to/vllm_omni_env/bin/activate
> pip install -e examples/tts/easymagpie_vllm_omni
> ```

In [ ]:
import os

# Single-process executor below, but keep spawn semantics consistent with the
# qwen3-tts / eartts demos in case you switch to a multiproc backend.
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

import json
import tempfile
import uuid
from pathlib import Path

import torch
import yaml

from vllm import SamplingParams
from vllm_omni import AsyncOmni

# Importing the model package is optional (the engine resolves the arch via the
# `vllm.general_plugins` entry point installed with the package), but doing it
# here surfaces the arch dataclass we use to size the dummy prompt embedding.
from easymagpie_vllm_omni.config import EasyMagpieOmniArch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. Build a tiny dummy model directory

The engine only needs a `config.json` that (a) names the registered arch and
(b) carries the EasyMagpie + Qwen2 scalars. We deliberately pick **small** dims
so the dummy backbone and local transformer are fast to instantiate.

The EasyMagpie-specific scalars (`embedding_dim`, `num_audio_codebooks`,
`codebook_size`, `frame_stacking_factor`, `local_transformer_*`, …) are read by
`EasyMagpieOmniArch.from_hf_config`; the standard Qwen2 fields (`hidden_size`,
`num_hidden_layers`, …) configure the reused `Qwen2Model` backbone. Setting
`phoneme_vocab_size = 0` disables the optional phoneme branch for simplicity.

With `load_format=\"dummy\"` (set in the stage config) vLLM never reads weight
files, so a lone `config.json` is enough — no safetensors, no tokenizer.

In [ ]:
# Small, internally-consistent dummy profile.
#   embedding_dim == hidden_size == audio_embedding_dim == local_transformer_hidden_dim
#   keeps every in/out projection an Identity (fewer dummy params, same code path).
HIDDEN = 256
NUM_AUDIO_CODEBOOKS = 4
CODEBOOK_SIZE = 64
FRAME_STACKING = 2  # -> num_stacked_codebooks = NUM_AUDIO_CODEBOOKS * FRAME_STACKING = 8
TEXT_VOCAB = 256

config = {
    # Resolved through the `vllm.general_plugins` entry point registered by the
    # `easymagpie_vllm_omni` package -> EasyMagpieTTSForConditionalGeneration.
    "architectures": ["EasyMagpieTTSForConditionalGeneration"],
    # Standard Qwen2 backbone fields (consumed by vllm Qwen2Model).
    "model_type": "qwen2",
    "hidden_size": HIDDEN,
    "intermediate_size": 4 * HIDDEN,
    "num_hidden_layers": 2,
    "num_attention_heads": 4,
    "num_key_value_heads": 4,
    "max_position_embeddings": 4096,
    "rms_norm_eps": 1e-6,
    "rope_theta": 1000000.0,
    "vocab_size": TEXT_VOCAB,
    "tie_word_embeddings": False,
    "torch_dtype": "float32",
    # EasyMagpie-specific scalars (read by EasyMagpieOmniArch.from_hf_config).
    "text_vocab_size": TEXT_VOCAB,
    "embedding_dim": HIDDEN,
    "audio_embedding_dim": HIDDEN,
    "num_audio_codebooks": NUM_AUDIO_CODEBOOKS,
    "codebook_size": CODEBOOK_SIZE,
    "frame_stacking_factor": FRAME_STACKING,
    "phoneme_stacking_factor": 0,  # disable phoneme branch
    "phoneme_vocab_size": 0,
    "local_transformer_n_layers": 2,
    "local_transformer_n_heads": 4,
    "local_transformer_hidden_dim": HIDDEN,
}

MODEL_DIR = Path(tempfile.mkdtemp(prefix="easymagpie_dummy_"))
(MODEL_DIR / "config.json").write_text(json.dumps(config, indent=2))
print(f"Dummy model dir: {MODEL_DIR}")

# Sanity-check the arch the model will derive from this config.
arch = EasyMagpieOmniArch.from_hf_config(type("Cfg", (), config))
print(f"embedding_dim            : {arch.embedding_dim}")
print(f"num_stacked_codebooks    : {arch.num_stacked_codebooks}  (C*S)")
print(f"tokens / codebook        : {arch.num_all_tokens_per_codebook}  (codebook_size + specials)")
print(f"audio_bos / audio_eos id : {arch.audio_bos_id} / {arch.audio_eos_id}")

## 2. Single-stage `AsyncOmni` engine

A single `llm` stage that runs the EasyMagpie talker, mirroring the eartts demo
(`worker_type=\"ar\"`, `OmniARScheduler`). The stage declares
`engine_output_type=\"audio\"` / `final_output_type=\"audio\"`: for a single-stage
AR TTS model these make the runner attach the per-step `audio_codes` multimodal
payload to the output (with `\"latent\"` the payload is dropped because nothing
downstream consumes it, and `multimodal_output[\"audio_codes\"]` comes back
`None`). Two extra knobs make this a dummy-weights run with no external assets:

* `load_format: \"dummy\"` — vLLM initializes random weights instead of reading a
  checkpoint (so `load_weights` / `init_forbidden_mask` are skipped; the
  forbidden-token mask stays all-zeros, i.e. no sampling mask — fine for a smoke
  test).
* `skip_tokenizer_init: true` — we feed `prompt_token_ids` + `text_tokens`
  directly, so no tokenizer files are needed.

`max_model_len` must cover `T_ctx` (prefill) + the number of decode steps.

In [ ]:
T_CTX = 16            # prefill context-embedding length (prompt_token_ids = [0] * T_CTX)
DECODE_STEPS = 32     # number of audio frames to decode
MAX_MODEL_LEN = 512
MAX_NUM_BATCHED_TOKENS = 512

stage_cfg = {
    "stage_args": [
        {
            "stage_id": 0,
            "stage_type": "llm",
            "is_comprehension": True,
            "final_output": True,
            # "audio" (not "latent") is required for a single-stage AR TTS model:
            # it makes the AR model runner attach the per-step multimodal payload
            # ("audio_codes") to the EngineCoreOutput even though no downstream
            # stage consumes it, so the codes reach the client. With "latent" the
            # payload is dropped and multimodal_output["audio_codes"] is None.
            "final_output_type": "audio",
            "runtime": {"devices": "0"},
            "engine_args": {
                "model_stage": "easymagpie",
                "max_num_seqs": 1,
                "model_arch": "EasyMagpieTTSForConditionalGeneration",
                "worker_type": "ar",
                "scheduler_cls": "vllm_omni.core.sched.omni_ar_scheduler.OmniARScheduler",
                "enforce_eager": True,  # dummy run: skip CUDA-graph capture for a faster start
                "trust_remote_code": True,
                "async_scheduling": True,
                "enable_prefix_caching": False,
                "engine_output_type": "audio",
                "gpu_memory_utilization": 0.6,
                "distributed_executor_backend": "uni",
                "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
                "max_model_len": MAX_MODEL_LEN,
                "dtype": "float32",
                "attention_backend": "TRITON_ATTN",
                # --- dummy-weights smoke-test knobs ---
                "load_format": "dummy",
                "skip_tokenizer_init": True,
            },
            "default_sampling_params": {
                "temperature": 0.0,
                "max_tokens": DECODE_STEPS,
                "detokenize": False,
                "ignore_eos": True,
            },
        }
    ],
}

_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", prefix="easymagpie_omni_demo_", delete=False,
)
yaml.dump(stage_cfg, _tmp, sort_keys=False)
_tmp.close()
STAGE_CFG_PATH = _tmp.name
print(f"Stage config: {STAGE_CFG_PATH}")

omni = AsyncOmni(
    model=str(MODEL_DIR),
    stage_configs_path=STAGE_CFG_PATH,
    log_stats=False,
    stage_init_timeout=300,
)
print("Engine ready (single stage: EasyMagpie talker, dummy weights)")

## 3. Build the prompt

Two pieces of per-request input, passed through `additional_information`:

* **`prompt_embeds`** `(T_ctx, embedding_dim)` — the precomputed context
  embedding consumed during prefill. In a real run this is the speaker-encoded
  context audio + context text produced by the caller; here we use random noise.
  `prompt_token_ids = [0] * T_ctx` are placeholders (the model feeds the backbone
  via `inputs_embeds`, never via these ids).
* **`text_tokens`** `list[int]` — the streaming subword stream; decode step `k`
  consumes `text_tokens[k]`. We provide one id per decode step.

(If the checkpoint had a phoneme branch you'd also stream `phoneme_tokens`; it's
disabled here via `phoneme_vocab_size = 0`.)

In [ ]:
torch.manual_seed(0)

# Precomputed context embedding (random stand-in for the speaker/text encoder).
prompt_embeds = torch.randn(T_CTX, arch.embedding_dim, dtype=torch.float32)

# Streaming subword ids: one per decode step (step k consumes text_tokens[k]).
text_tokens = torch.randint(0, TEXT_VOCAB, (DECODE_STEPS,)).tolist()

additional_information = {
    "prompt_embeds": prompt_embeds,   # (T_ctx, embedding_dim) tensor
    "text_tokens": text_tokens,       # list[int], grows by one per step
}

prompt = {
    "prompt_token_ids": [0] * T_CTX,  # prefill placeholders
    "additional_information": additional_information,
}

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=DECODE_STEPS,
    detokenize=False,
    ignore_eos=True,  # dummy logits never emit a meaningful EOS -> run the full budget
)

print(f"T_ctx (prefill placeholders) : {T_CTX}")
print(f"prompt_embeds                : {tuple(prompt_embeds.shape)}")
print(f"decode steps (max_tokens)    : {DECODE_STEPS}")
print(f"text_tokens[:8]              : {text_tokens[:8]}")

## 4. Run inference and extract audio codes

`omni.generate(...)` is an async generator yielding one `RequestOutput` per
engine step; we keep the last one. As in the eartts demo, the accumulated
`multimodal_output[\"audio_codes\"]` holds one row per flat-batch token over the
whole run (the `T_ctx` prefill frames — codes left zero — plus one frame per
decode step), so we trim to the last `len(token_ids)` rows to recover just the
decoded frames.

In [ ]:
async def run_request(prompt: dict, sampling_params):
    request_id = f"easymagpie-{uuid.uuid4().hex[:8]}"
    final_ro = None
    num_steps = 0
    async for stage_output in omni.generate(
        prompt,
        sampling_params_list=[sampling_params],
        request_id=request_id,
    ):
        final_ro = stage_output
        num_steps += 1
    return final_ro, num_steps


final_ro, num_steps = await run_request(prompt, sampling_params)
assert final_ro is not None, "no output from engine"

mm = final_ro.multimodal_output or {}
audio_codes = mm.get("audio_codes")
token_ids = final_ro.outputs[0].token_ids if final_ro.outputs else []

print(f"Engine steps yielded       : {num_steps}")
print(f"Layer-0 tokens (token_ids) : {len(token_ids)}")
if isinstance(audio_codes, torch.Tensor):
    audio_codes = audio_codes.detach().cpu().to(torch.long)
    print(f"audio_codes shape (raw)    : {tuple(audio_codes.shape)}")
    # Trim the Tref prefill frames echoed during prefill: keep only the decoded
    # frames (the last len(token_ids) rows), exactly like the eartts demo.
    if len(token_ids) > 0:
        audio_codes = audio_codes[-len(token_ids):].contiguous()
    print(f"audio_codes shape (decode) : {tuple(audio_codes.shape)}")
    print(f"audio_codes dtype          : {audio_codes.dtype}")
    print(f"codes min / max            : {int(audio_codes.min())} / {int(audio_codes.max())}")
    print(f"first frame codes          : {audio_codes[0].tolist()}")
else:
    print(f"audio_codes                : {audio_codes!r}")

In [ ]:
import matplotlib.pylab as plt

plt.imshow(audio_codes.T, aspect="auto")
plt.colorbar()
plt.show()